# Data Gathering Walkthrough

Welcome to this **expanded** data gathering tutorial! We’ll cover how to:
1. **Read CSV files** (local and remote) with basic options.
2. **Parse dates** when reading CSVs.
3. **Concatenate multiple CSV files**.
4. **Fetch data from an API** (JSON) and create a Pandas DataFrame.
5. **Make a second API call** to demonstrate parameters.
6. **Scrape a webpage** using **BeautifulSoup**.

This notebook is still aimed at **beginners**—with just enough extra features to be fun. Let's get started!

## 1. Installing and Importing Libraries

We'll use:
- **Pandas** for reading CSV files and handling data.
- **Requests** for making HTTP requests to APIs or websites.
- **BeautifulSoup** (from `bs4`) for **web scraping** HTML.

If you don’t have these installed locally, use:
```bash
pip install pandas requests beautifulsoup4
```
In Google Colab, we can run `!pip install` in a cell (shown below).

In [10]:
import sys
!{sys.executable} -m pip install --quiet pandas requests beautifulsoup4

Now we import the libraries we'll use.

In [11]:
import pandas as pd
import requests
import numpy as np
from bs4 import BeautifulSoup

# Adjust some display settings so tables look nicer.
#pd.set_option('display.max_rows', 6)
#pd.set_option('display.max_columns', 6)
#pd.set_option('display.width', 100)

## 2. Reading CSV Files

### 2.1 Local CSV File (Optional)
If you have a CSV file on your local machine, you can use `pd.read_csv("path/to/file.csv")` to load it. In Colab, you might upload the file via the left panel or `files.upload()`.

The code below **won’t** run unless you replace `'local_data.csv'` with a real filename you have in the same directory. We'll just show how it works in principle:

In [14]:
csv_path = '/content/sample_data/california_housing_test.csv'  # Replace with your real local CSV file /content/sample_data/california_housing_test.csv
try:
    local_df = pd.read_csv(csv_path)
    display(local_df.head())
except FileNotFoundError:
    print(f"'{csv_path}' not found. Replace with your own CSV path.")

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-122.05,37.37,27.0,3885.0,661.0,1537.0,606.0,6.6085,344700.0
1,-118.30,34.26,43.0,1510.0,310.0,809.0,277.0,3.5990,176500.0
2,-117.81,33.78,27.0,3589.0,507.0,1484.0,495.0,5.7934,270500.0
3,-118.36,33.82,28.0,67.0,15.0,49.0,11.0,6.1359,330000.0
4,-119.67,36.33,19.0,1241.0,244.0,850.0,237.0,2.9375,81700.0


### 2.2 Remote CSV File
You can load a CSV **directly from a URL** if it's publicly accessible. Below, we’ll pull in a classic [Titanic dataset](https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv) from GitHub.

In [15]:
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic_df = pd.read_csv(url)
titanic_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### 2.3 Parsing Dates
Some CSV columns contain dates (e.g., `'2023-01-01'`). Pandas can parse them automatically using `parse_dates=["ColumnName"]`.

Below is a demonstration using a CSV with a date column. We’ll just show the code; if the CSV doesn’t actually have date columns, adapt it as needed.

In [16]:
dates_csv_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'
try:
    # This CSV has a 'Date' column we can parse.
    temps_df = pd.read_csv(dates_csv_url, parse_dates=['Date'])
    display(temps_df.head())
    print("\nData types:")
    print(temps_df.dtypes)
except Exception as e:
    print("Error reading or parsing the CSV:", e)

,Date,Temp
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8



Data types:
Date    datetime64[ns]
Temp           float64
dtype: object


### 2.4 Concatenating Multiple CSV Files
If you have multiple CSVs with **identical columns**, you might want to combine them. Here's a simple example using two small DataFrames (pretending they came from two CSV files) and **concatenating** them row-wise.

In [18]:
# Pretend these are read from separate CSV files.
df_a = pd.DataFrame({
    'Col1': [1, 2],
    'Col2': ['A', 'B']
})
df_b = pd.DataFrame({
    'Col1': [3, 4],
    'Col2': ['C', 'D']
})

combined_df = pd.concat([df_a, df_b], axis=0, ignore_index=True)
print("DataFrame A:")
display(df_a)
print("DataFrame B:")
display(df_b)
print("Combined:")
display(combined_df)

DataFrame A:


,Col1,Col2
0,1,A
1,2,B


DataFrame B:


,Col1,Col2
0,3,C
1,4,D


Combined:


,Col1,Col2
0,1,A
1,2,B
2,3,C
3,4,D


## 3. Making an API Request

We’ll use the **Requests** library to call a simple API, then convert the JSON to a Pandas DataFrame.

### 3.1 Simple API Example
We’ll use [JSONPlaceholder](https://jsonplaceholder.typicode.com/) again, which provides dummy data for testing. We'll fetch a list of *users* in JSON format and convert them to a DataFrame.

In [19]:
api_url = 'https://jsonplaceholder.typicode.com/users'
response = requests.get(api_url)

if response.status_code == 200:
    data_json = response.json()  # This is a list of dictionaries
    users_df = pd.DataFrame(data_json)
    print("Successfully fetched", len(users_df), "users!")
    display(users_df)
else:
    print("Request failed with status code:", response.status_code)

Successfully fetched 10 users!


,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."
5,6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,"{'street': 'Norberto Crossing', 'suite': 'Apt....",1-477-935-8478 x6430,ola.org,"{'name': 'Considine-Lockman', 'catchPhrase': '..."
6,7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,"{'street': 'Rex Trail', 'suite': 'Suite 280', ...",210.067.6132,elvis.io,"{'name': 'Johns Group', 'catchPhrase': 'Config..."
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,"{'street': 'Ellsworth Summit', 'suite': 'Suite...",586.493.6943 x140,jacynthe.com,"{'name': 'Abernathy Group', 'catchPhrase': 'Im..."
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,"{'street': 'Dayna Park', 'suite': 'Suite 449',...",(775)976-6794 x41206,conrad.com,"{'name': 'Yost and Sons', 'catchPhrase': 'Swit..."
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,"{'street': 'Kattie Turnpike', 'suite': 'Suite ...",024-648-3804,ambrose.net,"{'name': 'Hoeger LLC', 'catchPhrase': 'Central..."


Note that some columns in `users_df` might be nested dictionaries (e.g., `address`, `company`). You can expand them by creating new columns or using advanced methods like `pandas.json_normalize`.

### 3.2 Second API Call (Optional)
Sometimes you need to **pass parameters** or **headers**. Here’s a quick example calling the `posts` endpoint from JSONPlaceholder with a query parameter (`userId`).

In [20]:
posts_url = 'https://jsonplaceholder.typicode.com/posts'
params = {
    'userId': 1  # only fetch posts for userId=1
}

posts_response = requests.get(posts_url, params=params)
if posts_response.status_code == 200:
    posts_json = posts_response.json()
    posts_df = pd.DataFrame(posts_json)
    print(f"Fetched {len(posts_df)} posts for userId=1.")
    display(posts_df.head())
else:
    print("Request failed with status code:", posts_response.status_code)

Fetched 10 posts for userId=1.


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


## 4. Basic Web Scraping with BeautifulSoup

Web scraping allows you to extract data directly from **HTML web pages**. However, always check the site's **Terms of Service** before scraping.

We'll demonstrate scraping [Quotes to Scrape](http://quotes.toscrape.com/), a test site specifically designed for practicing web scraping. We'll:
1. **Fetch** the page HTML with `requests.get`.
2. **Parse** the HTML using **BeautifulSoup**.
3. **Extract** the quote text and the author from the page.
4. Store the results in a **DataFrame**.

In [21]:
scrape_url = 'http://quotes.toscrape.com/'
response_scrape = requests.get(scrape_url)
if response_scrape.status_code == 200:
    soup = BeautifulSoup(response_scrape.text, 'html.parser')

    # Find all quote blocks on the page.
    quote_blocks = soup.find_all('div', class_='quote')

    quotes_data = []
    for qb in quote_blocks:
        text = qb.find('span', class_='text').get_text(strip=True)
        author = qb.find('small', class_='author').get_text(strip=True)
        quotes_data.append({
            'quote': text,
            'author': author
        })

    quotes_df = pd.DataFrame(quotes_data)
    print("Scraped", len(quotes_df), "quotes!")
    display(quotes_df)
else:
    print("Error fetching the webpage.")

Scraped 10 quotes!


,quote,author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe
5,“Try not to become a man of success. Rather be...,Albert Einstein
6,“It is better to be hated for what you are tha...,André Gide
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin


### Extracting More Data
You can scrape additional info (e.g., tags associated with each quote) by locating them in the HTML and repeating the process. Some pages are more complex or use JavaScript to load data, in which case you might need additional tools.

Always remember to **respect robots.txt** and site **ToS**.

## 5. Challenge: Try It Yourself!

1. **CSV with Dates**: Find a CSV that has a date column (e.g., a historical stock price dataset). Use `parse_dates=["column"]` to parse the dates.
2. **Multiple CSV**: Concatenate or merge two CSV files that share similar columns.
3. **API**: Try a different endpoint on [JSONPlaceholder](https://jsonplaceholder.typicode.com/) (e.g., `/comments`) or another public API. Parse the JSON into a DataFrame.
4. **Scraping**: On [Quotes to Scrape](http://quotes.toscrape.com/) (or another site), extract more info (e.g., tags). Alternatively, find a site with a table and scrape that table into a DataFrame.

When you’re done, share your findings or compare with classmates!

## 6. Summary

In this **expanded** tutorial, you learned:

1. Reading CSV files (local and remote), including **date parsing**.
2. **Concatenating** multiple CSV files.
3. Gathering data from **APIs** (JSONPlaceholder) and using parameters.
4. **Web scraping** a simple website with **Requests** + **BeautifulSoup**.

With these skills, you can gather data from a wide range of sources—files, APIs, and even HTML pages. Enjoy exploring your data!